# How to create a Trend Following strategy

![My Image](https://drive.google.com/uc?export=view&id=1eSJfk1esvGIH14m2n0Z-jVAQYmGQ7uby)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Sample OHLC data ships with the repo (data/samples/). Resolve it whether the
# notebook runs from the repo root (VS Code default) or from its own folder.
csv_name = 'momentum_ohlc.csv'
csv_path = next(p for p in (Path('data/samples') / csv_name,
                            Path('../../data/samples') / csv_name) if p.exists())
df = pd.read_csv(csv_path)
df = df.set_index('t')
df


## Add Log Returns and auto-regressive lag

In [ ]:
df['close_log_return'] = np.log(df['c']/df['c'].shift())
df['close_log_return_lag_1'] = df['close_log_return'].shift()
df

In [ ]:
df['close_log_return_dir_lag_1'] = np.sign(df['close_log_return_lag_1'])
df

## Research Price Movements

In [ ]:
df[['close_log_return','close_log_return_lag_1']].corr()

In [ ]:
df.groupby('close_log_return_dir_lag_1').aggregate({'close_log_return':['sum','mean','count']})

## Do Time Split

In [ ]:
i = int(len(df) * 0.75)
in_sample, out_sample = df.iloc[:i], df.iloc[i:]



In [ ]:
in_sample

In [ ]:
out_sample

In [ ]:
in_sample.groupby('close_log_return_dir_lag_1').aggregate({'close_log_return':['sum','mean','count']})

In [ ]:
out_sample.groupby('close_log_return_dir_lag_1').aggregate({'close_log_return':['sum','mean','count']})

## Add Backtest

In [ ]:
df['signal'] = df['close_log_return_dir_lag_1']

In [ ]:
df

In [ ]:
df['trade_log_return'] = df['close_log_return'] * df['signal']
df

In [ ]:
df['cum_trade_log_return'] = df['trade_log_return'].cumsum()
df

In [ ]:
df['cum_trade_log_return'].plot()

## Strategy Statistics

In [ ]:
df['is_won'] = df['trade_log_return'] > 0
df['is_won'].mean()

In [ ]:
df['trade_log_return'].mean()

In [ ]:
df['trade_log_return'].std()

### Annualized Sharpe

This dataset uses **weekly (1w)** bars, so there are `365 / 7 ≈ 52` periods per year. The per-period Sharpe is scaled by `sqrt(52)` to annualize.


In [ ]:
df['trade_log_return'].mean() / df['trade_log_return'].std() * np.sqrt(365 / 7)

### Add Fees

In [ ]:
CAPITAL = 1000
df['post_trade_notional_value'] = CAPITAL + CAPITAL * df['cum_trade_log_return']
df

In [ ]:
df['pre_trade_notional_value'] = df['post_trade_notional_value'].shift()
df

In [ ]:
df['pre_trade_notional_value'] = df['pre_trade_notional_value'].fillna(CAPITAL)
df

In [ ]:
TAKER_FEE_BPS = 4.1
MAKER_FEE_BPS = 1.2

TAKER_FEE_PCT = TAKER_FEE_BPS / 10000
MAKER_FEE_PCT = MAKER_FEE_BPS / 10000

df['entry_fee'] = df['pre_trade_notional_value'] * TAKER_FEE_PCT
df['exit_fee'] = df['post_trade_notional_value'] * TAKER_FEE_PCT
df['tx_fees'] = df['entry_fee'] + df['exit_fee']
df

In [ ]:
df['cum_tx_fees'] = df['tx_fees'].cumsum()
df

In [ ]:
df['net_equity'] = df['post_trade_notional_value'] - df['cum_tx_fees']
df

In [ ]:
df['net_equity'].plot()

In [ ]:
df['cum_trade_log_return'].plot()